In [ ]:
import datasets
import pandas as pd
import transformers

import shap

shap.initjs()

# load the emotion dataset
dataset = datasets.load_dataset("emotion", split="train")
data = pd.DataFrame({"text": dataset["text"], "emotion": dataset["label"]})

In [ ]:
from tklearn.kb import KnowledgeBase

kb = KnowledgeBase("wiktionary")

In [ ]:
# load the model and tokenizer
tokenizer = transformers.AutoTokenizer.from_pretrained(
    "nateraw/bert-base-uncased-emotion", use_fast=True
)
model = transformers.AutoModelForSequenceClassification.from_pretrained(
    "nateraw/bert-base-uncased-emotion"
).to("mps")

# build a pipeline object to do predictions
pipeline = transformers.pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device="mps",
    top_k=None,
)

In [ ]:
data

In [ ]:
import pandas as pd

preds = []
for i in range(data.shape[0]):
    for augment in kb.augment(data["text"][i], mentions=None):
        text = augment["text"]
        p = pipeline(text)
        preds.append({
            "idx": i,
            **augment,
            "labels": {item["label"]: item["score"] for item in p[0]},
        })

In [ ]:
from datasets import Dataset

df = pd.DataFrame(preds)

In [ ]:
df[df["support"] == 0]

In [ ]:
labels_df = df["labels"].apply(pd.Series)

labels_cols = labels_df.columns.tolist()

df = pd.concat([df, labels_df], axis=1)

In [ ]:
df.head(1)

In [ ]:
from scipy.spatial.distance import cdist

In [ ]:
df["score"] = 0.0

for i in df.idx.unique():
    original_labels_df = labels_df.loc[
        (df.idx == i) & (df.support == 0), labels_cols
    ]
    augmented_labels_df = labels_df.loc[
        (df.idx == i) & (df.support != 0), labels_cols
    ]
    d = cdist(augmented_labels_df, original_labels_df, metric="jensenshannon")
    scores = (1 - d[:, 0]).round(2)
    df.loc[(df.idx == i) & (df.support != 0), "score"] = scores

In [ ]:
df

In [ ]:
temp_df = (
    df[["relations", "score"]]
    .explode("relations")
    .dropna()
    .groupby("relations")
    .mean()
    .reset_index()
    .sort_values("score", ascending=False)
)

In [ ]:
# select top 10 and bottom 10 relations
top_relations = temp_df.head(10)
bottom_relations = temp_df.tail(10)
pd.concat([top_relations, bottom_relations]).reset_index(drop=True)